# 01 - Exploratory Data Analysis

Marketing Mix Modeling dataset from Conjura (132K+ rows, ~100 eCommerce brands).

In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

df = pl.read_parquet('../data/processed/mmm_cleaned.parquet')
print(f"Rows: {df.height:,}, Columns: {df.width}")

## Brand & Territory Distribution

In [ ]:
brand_counts = df.group_by('organisation_vertical').agg(pl.len().alias('count')).sort('count', descending=True)
fig = px.bar(brand_counts.to_pandas(), x='organisation_vertical', y='count', title='Brands by Vertical')
fig.show()

## Total Spend Over Time

In [ ]:
daily = df.group_by('date_day').agg(pl.sum('total_spend').alias('total_spend'))
fig = px.line(daily.to_pandas(), x='date_day', y='total_spend', title='Daily Total Ad Spend')
fig.show()

## Channel Spend Distribution

In [ ]:
spend_cols = [c for c in df.columns if c.endswith('_spend')]
channel_spend = {c.replace('_spend', ''): df[c].sum() for c in spend_cols if df[c].sum() > 0}
spend_df = pl.DataFrame({'channel': list(channel_spend.keys()), 'spend': list(channel_spend.values())})
fig = px.pie(spend_df.to_pandas(), values='spend', names='channel', title='Total Spend by Channel')
fig.show()

## Revenue vs Spend Correlation

In [ ]:
sample = df.sample(5000)
fig = px.scatter(sample.to_pandas(), x='total_spend', y='first_purchases_original_price',
                 title='Total Spend vs First-Purchase Revenue', opacity=0.5)
fig.show()

## CTR Distribution by Channel

In [ ]:
ctr_cols = [c for c in df.columns if c.endswith('_ctr')]
ctr_data = []
for c in ctr_cols:
    for val in df.filter(pl.col(c) > 0)[c].to_list():
        ctr_data.append({'channel': c.replace('_ctr', ''), 'ctr': val})
ctr_df = pl.DataFrame(ctr_data)
fig = px.box(ctr_df.to_pandas(), x='channel', y='ctr', title='CTR Distribution by Channel')
fig.update_xaxes(tickangle=45)
fig.show()